# GYMTRACE — Dataset Cleaning and Pre-Processing

Dataset: Crowdedness at the Campus Gym (Kaggle)  
Target: `number_people`

Upload the dataset zip or `data.csv` in the next cell, then run the rest of the notebook.

## 1. Upload dataset

Upload either:
- the Kaggle zip file, or
- `data.csv` directly

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
import io

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

work = Path("/content/gymtrace")
raw_dir = work / "data" / "raw"
processed_dir = work / "data" / "processed"
proof_dir = work / "proof"

for folder in [raw_dir, processed_dir, proof_dir]:
    folder.mkdir(parents=True, exist_ok=True)

print("Choose your zip or data.csv")
uploaded = files.upload()

raw_csv = raw_dir / "data.csv"

for name, content in uploaded.items():
    lower = name.lower()
    if lower.endswith(".zip"):
        with zipfile.ZipFile(io.BytesIO(content)) as zf:
            zf.extractall(raw_dir)
        csv_files = list(raw_dir.glob("*.csv"))
        if not csv_files:
            raise FileNotFoundError("No CSV found inside the zip.")
        if raw_csv.exists():
            raw_csv.unlink()
        csv_files[0].rename(raw_csv)
        print("Extracted", csv_files[0].name, "→", raw_csv)
    elif lower.endswith(".csv"):
        raw_csv.write_bytes(content)
        print("Saved uploaded CSV →", raw_csv)

if not raw_csv.exists():
    raise FileNotFoundError("Upload a .zip or data.csv first.")

print("Ready:", raw_csv, "| size MB:", round(raw_csv.stat().st_size / 1e6, 2))

## 2. Load and inspect (before cleaning)

In [ ]:
df = pd.read_csv(raw_csv)

print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head(10)

In [ ]:
df.info()
df.describe()

In [ ]:
def save_table(frame, title, path):
    show = frame.copy()
    # shorten long date strings so the table fits
    for col in show.columns:
        show[col] = show[col].astype(str).str.slice(0, 22)
    fig, ax = plt.subplots(figsize=(14, 3.5))
    ax.axis("off")
    ax.set_title(title, loc="left", fontsize=12, pad=8)
    table = ax.table(
        cellText=show.values,
        colLabels=show.columns,
        loc="center",
        cellLoc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(7)
    table.scale(1.05, 1.35)
    plt.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved", path)

save_table(
    df.head(8),
    "Before cleaning — first 8 rows",
    proof_dir / "03_before_head_table.png",
)

## 3. Check problems in the data

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
duplicates = int(df.duplicated().sum())

print("Missing values per column:")
print(missing)
print("\nTotal missing cells:", int(missing.sum()))
print("Duplicate rows:", duplicates)
print("\nData types:")
print(df.dtypes)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
missing.plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Missing values per column")
ax.set_ylabel("count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
fig.savefig(proof_dir / "01b_during_missing_values.png", dpi=150)
plt.show()
print("Saved", proof_dir / "01b_during_missing_values.png")

In [ ]:
def iqr_bounds(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for col in ["number_people", "temperature"]:
    low, high = iqr_bounds(df[col])
    outs = int(((df[col] < low) | (df[col] > high)).sum())
    print(f"{col}: IQR range [{low:.2f}, {high:.2f}], outliers = {outs}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(df["number_people"], bins=40, color="steelblue", edgecolor="white")
axes[0].set_title("number_people (before)")
axes[0].set_xlabel("people")
axes[1].boxplot(df["number_people"])
axes[1].set_title("number_people boxplot (before)")
plt.tight_layout()
fig.savefig(proof_dir / "01_before_target_distribution.png", dpi=150)
plt.show()
print("Saved", proof_dir / "01_before_target_distribution.png")

## 4. Clean the data

Steps used:
- parse dates
- remove duplicate rows
- keep flag columns as 0/1
- fill missing numeric values with the median
- cap extreme `number_people` values using IQR
- clip unrealistic temperatures

In [ ]:
df_clean = df.copy()
before_rows = len(df_clean)

df_clean["date"] = pd.to_datetime(df_clean["date"], utc=True, errors="coerce")
df_clean = df_clean.drop_duplicates()

for col in ["is_weekend", "is_holiday", "is_start_of_semester", "is_during_semester"]:
    df_clean[col] = df_clean[col].astype(int)

for col in df_clean.select_dtypes(include=[np.number]).columns:
    if df_clean[col].isna().any():
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

low, high = iqr_bounds(df_clean["number_people"])
capped = int(((df_clean["number_people"] < low) | (df_clean["number_people"] > high)).sum())
df_clean["number_people"] = df_clean["number_people"].clip(lower=max(0, low), upper=high)
df_clean["temperature"] = df_clean["temperature"].clip(lower=20, upper=110)
df_clean = df_clean.dropna(subset=["date"])

after_rows = len(df_clean)
print("Rows before:", before_rows)
print("Rows after:", after_rows)
print("Rows removed:", before_rows - after_rows)
print("number_people values capped:", capped)
print("Missing left:", int(df_clean.isna().sum().sum()))

## 5. Prepare features for machine learning

In [ ]:
features = [
    "timestamp", "day_of_week", "is_weekend", "is_holiday",
    "temperature", "is_start_of_semester", "is_during_semester",
    "month", "hour",
]

X = df_clean[features]
y = df_clean["number_people"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=features,
    index=X_train.index,
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=features,
    index=X_test.index,
)

print("Train:", X_train.shape, "Test:", X_test.shape)
X_train_scaled.head()

## 6. Results after cleaning

In [ ]:
print("Cleaned shape:", df_clean.shape)
df_clean.head(10)

In [ ]:
df_clean.describe()

In [ ]:
save_table(
    df_clean.head(8),
    "After cleaning — first 8 rows",
    proof_dir / "04_after_head_table.png",
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(df["number_people"], bins=40, alpha=0.55, label="before", color="salmon")
axes[0].hist(df_clean["number_people"], bins=40, alpha=0.7, label="after", color="teal")
axes[0].set_title("number_people before vs after")
axes[0].legend()
axes[1].bar(["before", "after"], [len(df), len(df_clean)], color=["salmon", "teal"])
axes[1].set_title("Row count before vs after")
plt.tight_layout()
fig.savefig(proof_dir / "02_after_vs_before.png", dpi=150)
plt.show()

out_csv = processed_dir / "gym_crowdedness_clean.csv"
df_clean.to_csv(out_csv, index=False)
print("Saved cleaned CSV:", out_csv)

## 7. Download the figures

These files are what you can submit as proof:
- `01_before_target_distribution.png`
- `01b_during_missing_values.png`
- `02_after_vs_before.png`
- `03_before_head_table.png`
- `04_after_head_table.png`

In [ ]:
import shutil

print("Files in proof folder:")
for f in sorted(proof_dir.glob("*")):
    print(" -", f.name)

zip_name = "/content/gymtrace_cleaning_proof"
shutil.make_archive(zip_name, "zip", proof_dir)
files.download(zip_name + ".zip")
files.download(str(out_csv))